# E005 — spatial meniscus specialist

Kaggle-only frozen DINOv2 spatial-token pilot. No leaderboard submission.


In [ ]:
# E005 configuration: frozen DINOv2 spatial-region tokens for two meniscus targets.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import math
import random
import re
import shutil
import time
import unicodedata
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
SEED = 20260810
random.seed(SEED)
np.random.seed(SEED)

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]
SPECIAL_TARGETS = ["Medial Meniscus", "Lateral Meniscus"]

FOV_MM = 150.0
PATCH_PX = 14
EMBED_DIM = 384
N_BLOCKS = 12
IMG_SIZE = 336
SLICE_DIM = 3 * EMBED_DIM  # retained only for the inherited diagnostic helper

# Meniscal evidence is concentrated in sagittal/coronal fluid-sensitive series.
SLOT_SPEC = [("Sagittal", 1, 12), ("Coronal", 1, 8)]
SLOT_NAMES = ["{}_{}".format(p, "Fluid" if f else "Structural")
              for p, f, _ in SLOT_SPEC]
N_SLOTS = len(SLOT_SPEC)
MAX_SLICES = max(k for _, _, k in SLOT_SPEC)
PLANE_WINDOW = {"Sagittal": (0.08, 0.92), "Coronal": (0.15, 0.85),
                "Axial": (0.12, 0.88)}

# A 24x24 DINO patch grid is pooled to 8x8 spatial regions, then projected with a
# pinned orthogonal map. This preserves spatial layout while keeping the cache near
# 2.6 GB instead of storing roughly 75 GB of raw patch tokens.
REGION_GRID = 8
REGION_DIM = 192
REGIONS = REGION_GRID * REGION_GRID
SPATIAL_CACHE_VERSION = "meniscus_sagcor_fluid_g8_r192_v1"

MAX_HEADER_READS_PER_SERIES = 64
GPU_BATCH = 32
IO_WORKERS = max(2, min(8, (os.cpu_count() or 4)))
TEST_ENCODE_BUDGET_S = 6000.0
TRAIN_ENCODE_BUDGET_S = 12000.0

N_FOLDS = 5
HEAD_SEEDS = (20260810,)
HEAD_HIDDEN = 128
HEAD_HEADS = 4
HEAD_EPOCHS = 8
HEAD_BATCH = 16
HEAD_LR = 2.0e-3
HEAD_WD = 1.0e-2
HEAD_DROPOUT = 0.15

SPECIALIST_BLEND_W = 0.20
GATE_MEAN_GAIN = 0.003
GATE_LATERAL_GAIN = 0.005
GATE_MEDIAL_FLOOR = -0.003
GATE_GOLD_FLOOR = -0.020
GATE_MAX_CORR = 0.97

LABEL_SOURCE = "llm"
LLM_LABEL_PATTERNS = ["report_labels_v2.csv"]
RULE_LABEL_PATTERNS = ["labels_rule_*.parquet", "labels_rule_*.csv"]
LABEL_DATASET_SOURCES = ["pilkwang/rsna-knee-llm-labels"]
RULE_WEIGHT = 0.35
GOLD_WEIGHT = 3.0
EXPECTED_LABEL_SHA256 = "6f704a7bdb2f894cc49445b19ba7c4378c3f548d3449e00361e10044bee40920"
EXPECTED_ANCHOR_OOF_SHA256 = "62d47ba4c0c8347b5b24e7fd2aa517aae0fd6d4656fd5d829fd3a50f0159909c"

print("E005 spatial meniscus specialist")
print("slots:", list(zip(SLOT_NAMES, [x[2] for x in SLOT_SPEC])))
print("input: {} px | patch grid 24x24 -> region grid {}x{} | cache dim {}".format(
    IMG_SIZE, REGION_GRID, REGION_GRID, REGION_DIM))


In [ ]:
def detect_input_root():
    """Find the directory holding train.csv, without hardcoding a mount path.

    Returns:
        Path to the competition input root.

    Raises:
        FileNotFoundError: If no candidate directory contains train.csv.
    """
    # Kaggle does not always mount a competition at /kaggle/input/<slug>. For this one
    # it is /kaggle/input/competitions/<slug>, one level deeper, which a shallow scan
    # of /kaggle/input misses: it finds the `competitions` directory, sees no train.csv
    # inside it, and gives up. Version 1 of the companion baseline notebook died exactly
    # that way in public. So both shapes are named explicitly and the fallback walks two
    # levels rather than one.
    candidates = [
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/mnt/data/Github/Kaggle/Competitions/rsna_knee_abnormality_detection"
             "/data/kaggle_mirror/rsna-knee-abnormality-detection"),
        Path("/mnt/data/Github/Kaggle/Competitions/rsna_knee_abnormality_detection/data/raw"),
        Path("data/raw"),
        Path("."),
    ]
    for c in candidates:
        try:
            if (c / "train.csv").is_file():
                return c
        except OSError:
            continue

    kin = Path("/kaggle/input")
    seen = []
    if kin.is_dir():
        for depth1 in sorted(p for p in kin.iterdir() if p.is_dir()):
            seen.append(str(depth1))
            try:
                if (depth1 / "train.csv").is_file():
                    return depth1
                for depth2 in sorted(p for p in depth1.iterdir() if p.is_dir()):
                    seen.append(str(depth2))
                    if (depth2 / "train.csv").is_file():
                        return depth2
            except OSError:
                continue

    # Name what was checked AND what is actually mounted. A rerun failure is only
    # debuggable from the log, and "not found" on its own says nothing.
    raise FileNotFoundError(
        "no input root containing train.csv was found. Checked: "
        + ", ".join(str(c) for c in candidates)
        + ". Directories present under /kaggle/input: "
        + (", ".join(seen) if seen else "none"))


def find_subdir(root, names):
    """Return the first existing subdirectory of root among names, else None.

    Args:
        root: Directory to look inside.
        names: Candidate directory names, tried in order.

    Returns:
        A Path, or None when none of the names is a directory.
    """
    for n in names:
        p = root / n
        try:
            if p.is_dir():
                return p
        except OSError:
            continue
    return None


def find_dinov2_dir():
    """Locate the mounted DINOv2 checkpoint directory.

    The Kaggle model mount for metaresearch/dinov2/PyTorch/small/1 lands under
    /kaggle/input/dinov2/..., but the exact casing and the trailing version
    directory have moved before. So the explicit paths are tried first and then
    /kaggle/input is walked for any directory holding a config.json whose
    model_type is dinov2 next to a weights file. train_series and test_series are
    pruned from the walk; they hold hundreds of thousands of files.

    Returns:
        A Path to the checkpoint directory, or None when nothing matches.
    """
    explicit = [
        Path("/kaggle/input/dinov2/pytorch/small/1"),
        Path("/kaggle/input/dinov2/PyTorch/small/1"),
        Path("/kaggle/input/dinov2/transformers/small/1"),
    ]
    for p in explicit:
        try:
            if (p / "config.json").is_file():
                return p
        except OSError:
            continue

    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    skip = {"train_series", "test_series", "train_images", "test_images",
            ".git", "__pycache__"}
    best = None
    base_depth = len(base.parts)
    for root, dirs, files in os.walk(base):
        depth = len(Path(root).parts) - base_depth
        dirs[:] = [d for d in dirs if d not in skip and depth < 6]
        if "config.json" not in files:
            continue
        rp = Path(root)
        if not any((rp / w).is_file() for w in
                   ("model.safetensors", "pytorch_model.bin")):
            continue
        try:
            cfg = json.loads((rp / "config.json").read_text())
        except Exception:
            continue
        if "dinov2" not in str(cfg.get("model_type", "")).lower() \
                and "dinov2" not in str(rp).lower():
            continue
        score = abs(int(cfg.get("hidden_size", 10000)) - EMBED_DIM)
        if best is None or score < best[0]:
            best = (score, rp)
    return best[1] if best else None


ROOT = detect_input_root()
ON_KAGGLE = str(ROOT).startswith("/kaggle/")
TRAIN_DCM = find_subdir(ROOT, ["train_series", "train_images"])
TEST_DCM = find_subdir(ROOT, ["test_series", "test_images"])
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")

train = pd.read_csv(ROOT / "train.csv")
test = pd.read_csv(ROOT / "test.csv")


def read_optional(path, columns):
    """Read a CSV that may be absent, returning an empty typed frame instead.

    Args:
        path: CSV path.
        columns: Column names for the empty fallback frame.

    Returns:
        A DataFrame, empty with the given columns when the file is missing.
    """
    try:
        if path.is_file():
            return pd.read_csv(path)
    except Exception as exc:
        print("could not read", path.name, ":", exc)
    return pd.DataFrame(columns=columns)


SERIES_COLS = ["StudyInstanceUID", "SeriesInstanceUID", "Fluid_Sensitive",
               "Fat_Suppression", "Anatomical_Plane"]
train_series = read_optional(ROOT / "train_series.csv", SERIES_COLS)
test_series = read_optional(ROOT / "test_series.csv", SERIES_COLS)
sample_sub = read_optional(ROOT / "sample_submission.csv",
                           ["StudyInstanceUID"] + LABELS)

# ---- Torch and the backbone --------------------------------------------------
HAVE_TORCH = False
DEVICE = "cpu"
N_GPU = 0
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAVE_TORCH = True
    N_GPU = torch.cuda.device_count()
    DEVICE = "cuda" if N_GPU > 0 else "cpu"
    torch.manual_seed(SEED)
    if N_GPU > 0:
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = True
except Exception as exc:
    print("torch unavailable:", exc)

HAVE_PYDICOM = False
try:
    import pydicom
    HAVE_PYDICOM = True
except Exception as exc:
    print("pydicom unavailable:", exc)

# The resize backend is environment detection, not preprocessing: the resolution
# probe in section 2 needs it before section 6 exists. cv2 is present on Kaggle and
# absent in some local environments, so the helper degrades rather than failing.
try:
    import cv2
except Exception:
    cv2 = None
try:
    from PIL import Image
except Exception:
    Image = None


def resize_2d(arr, size):
    """Resize a 2D float array to size x size, with three fallbacks.

    cv2 is present on Kaggle and absent in some local environments, so the
    function degrades rather than failing: cv2, then Pillow's float mode, then a
    nearest-neighbour gather that needs nothing but NumPy.

    Args:
        arr: 2D float32 array.
        size: Target side length in pixels.

    Returns:
        A float32 array of shape (size, size).
    """
    h, w = arr.shape
    if (h, w) == (size, size):
        return arr.astype(np.float32)
    if cv2 is not None:
        interp = cv2.INTER_AREA if max(h, w) > size else cv2.INTER_LINEAR
        return cv2.resize(arr.astype(np.float32), (size, size),
                          interpolation=interp).astype(np.float32)
    if Image is not None:
        resample = Image.BILINEAR if max(h, w) <= size else Image.BOX
        return np.asarray(
            Image.fromarray(arr.astype(np.float32), mode="F")
            .resize((size, size), resample), dtype=np.float32)
    yi = np.clip((np.arange(size) * h / size).astype(int), 0, h - 1)
    xi = np.clip((np.arange(size) * w / size).astype(int), 0, w - 1)
    return arr[np.ix_(yi, xi)].astype(np.float32)


DINO_DIR = find_dinov2_dir()
CAN_ENCODE = bool(HAVE_TORCH and HAVE_PYDICOM and DINO_DIR is not None
                  and TRAIN_DCM is not None)

print("input root        :", ROOT)
print("running on        :", "Kaggle" if ON_KAGGLE else "local mirror")
print("train dicom dir   :", TRAIN_DCM)
print("test dicom dir    :", TEST_DCM)
print("writing to        :", WORK)
print("torch             :", "yes" if HAVE_TORCH else "NO",
      "| gpus:", N_GPU, "| device:", DEVICE)
print("pydicom           :", "yes" if HAVE_PYDICOM else "NO")
print("resize backend    :", "cv2" if cv2 is not None else
      ("PIL" if Image is not None else "numpy"))
print("dinov2 checkpoint :", DINO_DIR if DINO_DIR else "NOT FOUND")
print("encoding possible :", CAN_ENCODE)
print()
print("train studies     :", len(train), "| train series :", len(train_series))
print("test studies      :", len(test), "| test series  :", len(test_series))

gold_mask = (train[LABELS].notna().all(axis=1)
             if set(LABELS).issubset(train.columns)
             else pd.Series(False, index=train.index))
N_GOLD = int(gold_mask.sum())
gold_ids = train.loc[gold_mask, "StudyInstanceUID"].astype(str).tolist()
print("gold-labelled     :", N_GOLD, "of", len(train),
      "({:.2f}%)".format(100.0 * N_GOLD / max(1, len(train))))
if not CAN_ENCODE:
    print()
    print("NOTE: the imaging path cannot run here. Every later cell degrades to the")
    print("      series-manifest fallback and a valid submission is still written.")

In [ ]:
DINO = None
DINO_KW = {}
DINO_INFO = "not loaded"

if CAN_ENCODE:
    # This try covers the load and nothing else. An earlier version wrapped the
    # diagnostic prints below in the same block, and a NameError in one of those
    # prints set CAN_ENCODE=False and sent the whole notebook down the
    # metadata-only fallback while reporting "could not load DINOv2". A broad
    # except around code that is not the thing being guarded turns a typo into a
    # silent capability loss.
    try:
        import inspect
        from transformers import AutoModel
        DINO = AutoModel.from_pretrained(str(DINO_DIR), local_files_only=True)
        DINO.eval()
        for p in DINO.parameters():
            p.requires_grad_(False)
        DINO.to(DEVICE)
        # HF's Dinov2 embeddings interpolate the positional grid unconditionally,
        # but some releases also accept the flag explicitly. Decide from the
        # signature rather than from a try/except around the forward pass, which
        # would swallow a real error inside the model and report it as a
        # signature mismatch.
        if "interpolate_pos_encoding" in inspect.signature(DINO.forward).parameters:
            DINO_KW["interpolate_pos_encoding"] = True
    except Exception as exc:
        print("could not load DINOv2:", type(exc).__name__, exc)
        DINO = None
        CAN_ENCODE = False

if DINO is not None:
    try:
        n_par = sum(p.numel() for p in DINO.parameters())
        hidden = int(getattr(DINO.config, "hidden_size", EMBED_DIM))
        patch = int(getattr(DINO.config, "patch_size", PATCH_PX))
        native = int(getattr(DINO.config, "image_size", 0))
        DINO_INFO = "{} params, hidden {}, patch {}, configured at {} px".format(
            "{:,}".format(n_par), hidden, patch, native)
        print("DINOv2 loaded from", DINO_DIR)
        print("  ", DINO_INFO)
        print("   forward kwargs:", DINO_KW if DINO_KW else "none needed")
        if patch != PATCH_PX or hidden != EMBED_DIM:
            print("   WARNING: checkpoint geometry differs from the constants; the "
                  "millimetre arithmetic above assumed patch {} hidden {}".format(
                      PATCH_PX, EMBED_DIM))
        if native:
            print("   positional grid: native {0}x{0} -> {1}x{1} here, {2}x{2} at "
                  "the frontier's 126 px".format(
                      native // patch, IMG_SIZE // patch, 126 // patch))
    except Exception as exc:
        # Diagnostics only. Whatever fails here, the backbone is loaded and usable.
        print("checkpoint diagnostics unavailable:", type(exc).__name__, exc)

IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], np.float32).reshape(1, 3, 1, 1)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], np.float32).reshape(1, 3, 1, 1)


def embed_images(imgs):
    """Embed a batch of 2.5D images with the frozen backbone.

    Only cuda:0 is used even when two GPUs are visible. The GPU phase is a small
    share of this pipeline's wall clock and wrapping the model in DataParallel
    buys minutes at the cost of a code path that cannot be tested here.

    Args:
        imgs: uint8 array of shape (B, 3, size, size).

    Returns:
        float32 array of shape (B, SLICE_DIM), the CLS token concatenated with
        the mean and the maximum over patch tokens.
    """
    if DINO is None or imgs.shape[0] == 0:
        return np.zeros((imgs.shape[0], SLICE_DIM), np.float32)
    out = np.zeros((imgs.shape[0], SLICE_DIM), np.float32)
    with torch.no_grad():
        for start in range(0, imgs.shape[0], GPU_BATCH):
            chunk = imgs[start:start + GPU_BATCH]
            x = torch.from_numpy(chunk).to(DEVICE).float().div_(255.0)
            x = (x - torch.from_numpy(IMAGENET_MEAN).to(DEVICE)) \
                / torch.from_numpy(IMAGENET_STD).to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16,
                                enabled=(DEVICE == "cuda")):
                h = DINO(pixel_values=x, **DINO_KW).last_hidden_state
            h = h.float()
            vec = torch.cat([h[:, 0], h[:, 1:].mean(1), h[:, 1:].amax(1)], dim=1)
            out[start:start + chunk.shape[0]] = vec.cpu().numpy()
    return out

In [ ]:
# The rule extractor below is not written here. It is embedded verbatim from the
# builder module of the companion metadata-baseline notebook, so the two notebooks
# consume one extractor and cannot drift apart. Its audit against the gold studies
# is printed two cells down.
# --------------------------------------------------------------------------- #
# Multilingual rule-based label extractor.
# Weak on purpose. This is the artifact an LLM extraction has to beat.
# --------------------------------------------------------------------------- #

_CHAR_MAP = str.maketrans({"ı": "i", "İ": "i", "ß": "ss",
                           "ø": "o", "Ø": "o",
                           "đ": "d", "Đ": "d"})

TEAR = [
    "tear", "torn", "tearing", "rupture", "ruptured", "disruption", "discontinuity",
    "rotura", "ruptura", "desgarro", "roto", "rota",
    "dechirure", "dechire", "lesion meniscale",
    "scheur", "ruptuur", "gescheurd",
    "riss", "rissbildung", "ruptur", "zerreissung", "einriss",
    "yirtik", "yirtig", "kopma", "butunluk kaybi",
    "ρηξη", "ρηξις", "ρηγμα",
    "руптура", "разкъсв",
    "разрив", "puknuce", "prekid",
]
SPRAIN = [
    "sprain", "esguince", "entorse", "verstauchung", "verstuiking",
    "distorsiyon", "burkulma", "διαστρεμμα",
    "навяхван",
    "injury", "lesion", "letsel", "verletzung", "zedelenme",
]
OA_FIND = [
    "osteoarthritis", "osteoarthrosis", "arthrosis", "arthritic",
    "artrosis", "artrose", "arthrose", "gonartrose", "gonartroz", "gonarthrose",
    "gonartro", "artroz",
    "chondrosis", "chondral", "chondropathy", "chondropathie", "chondropatie",
    "chondromalacia", "kondromalazi", "condropatia", "condral", "chondropatia",
    "kraakbeenlijden", "kraakbeenverlies", "kraakbeenschade",
    "knorpelschaden", "knorpeldefekt", "knorpelverlust", "knorpelbelag",
    "cartilage loss", "cartilage thinning", "cartilage fissuring",
    "cartilage defect", "cartilage fissure", "chondral defect", "chondral loss",
    "osteophyt", "osteofit", "osteofyt", "osteophyte", "spurring", "osteofyte",
    "joint space narrowing", "gelenkspaltverschmalerung",
    "kikirdak kaybi", "kikirdak incelme", "kondral",
    "ulcera condral", "hrskavice", "denudacija",
    "χονδροπαθ", "οστεοαρθρ",
    "οστεοφυτ", "αρθριτ",
    "артроз", "хондропат",
    "остеофит", "хрущялн",
]
MEDIAL_Q = ["medial", "interno", "interna", "interne", "mediaal", "mediale",
            "innen", "medyal", "medijaln", "εσω",
            "медиал", "вътреш"]
LATERAL_Q = ["lateral", "externo", "externa", "externe", "buiten", "aussen",
             "lateraln", "εξω", "латерал",
             "външ"]
PF_Q = ["patellofemoral", "patelofemoral", "femoropatellar", "femoropatelar",
        "femoropatellair", "retropatellar", "retrorotulian", "patellar facet",
        "trochlea", "troclea", "trochlee", "rotula", "rotulian", "patella",
        "patellaire", "patellar", "diz kapagi", "patelofemoraln",
        "επιγονατιδ", "τροχιλ",
        "пател", "ретропател"]
TRICOMP = ["tricompartmental", "tricompartimental", "three compartments",
           "three compartmens", "all compartments", "gonartrose", "gonartroz",
           "gonarthrose", "gonartrosis", "gonartro", "pangonartro"]

SPECIFIC = {
    "ACL": [
        "acl", "anterior cruciate", "lca", "ligamento cruzado anterior",
        "ligament croise anterieur", "croise anterieur",
        "voorste kruisband", "vkb", "vorderes kreuzband", "vorderen kreuzband",
        "vordere kreuzband", "on capraz bag", "anterior capraz bag",
        "προσθιο χιαστ",
        "προσθιου χιαστ",
        "προσθιος χιαστ",
        "предна кръстна",
        "предната кръстна",
        "prednji ukrizeni",
    ],
    "MCL": [
        "mcl", "medial collateral", "lcm", "ligamento colateral medial",
        "ligamento colateral interno", "ligament collateral medial",
        "collateral medial", "mediale collaterale", "mediaal collateraal",
        "innenband", "mediales kollateralband", "medialen kollateralband",
        "mediale kollateralband", "medyal kollateral", "ic yan bag",
        "εσω πλαγιο",
        "медиален колатерален",
        "вътрешна колатерална",
    ],
    "Medial Meniscus": [
        "medial meniscus", "meniscus medialis", "menisco medial", "menisco interno",
        "mediale meniscus", "binnenmeniscus", "innenmeniskus", "meniskus medialis",
        "menisque interne", "menisque medial", "medial menisk", "medyal menisk",
        "εσω μηνισκ",
        "медиалния менискус",
        "медиален мениск",
        "вътрешния мениск",
        "medial and lateral menisc", "medial ve lateral menisk",
        "menisco medial y lateral", "menisco interno y externo",
        "menisco interno y lateral", "mediale en laterale meniscus",
        "innen und aussenmeniskus", "medijalnog meniskusa",
    ],
    "Lateral Meniscus": [
        "lateral meniscus", "meniscus lateralis", "menisco lateral", "menisco externo",
        "laterale meniscus", "buitenmeniscus", "aussenmeniskus", "meniskus lateralis",
        "menisque externe", "menisque lateral", "lateral menisk",
        "εξω μηνισκ",
        "латералния менискус",
        "латерален мениск",
        "външния мениск",
        "medial and lateral menisc", "medial ve lateral menisk",
        "menisco medial y lateral", "menisco interno y externo",
        "menisco interno y lateral", "mediale en laterale meniscus",
        "innen und aussenmeniskus", "lateralnog meniskusa",
    ],
}

GENERIC = {
    "ACL": ["cruciate", "cruzado", "croise", "kruisband", "kreuzband",
            "χιαστ", "кръстн",
            "capraz bag", "ukrizen"],
    "MCL": ["collateral", "colateral", "kollateral", "collaterale",
            "πλαγι", "колатерал",
            "yan bag"],
    "Medial Meniscus": ["menisc", "menisk", "μηνισκ",
                        "мениск"],
    "Lateral Meniscus": ["menisc", "menisk", "μηνισκ",
                         "мениск"],
}
SIDE_Q = {
    "ACL": ["anterior", "anterieur", "voorste", "vorder",
            "προσθι", "предн", "prednj"],
    "MCL": MEDIAL_Q,
    "Medial Meniscus": MEDIAL_Q,
    "Lateral Meniscus": LATERAL_Q,
}

DIRECT = {
    "Effusion": [
        "effusion", "joint fluid", "hemarthrosis", "haemarthrosis", "hydrops",
        "derrame", "epanchement", "gewrichtsvocht", "vocht in het gewricht",
        "erguss", "gelenkerguss", "gelenkserguss",
        "eklem ici sivi", "sivi artisi", "efuzyon", "eklem mesafesinde sivi",
        "eklem ici serbest sivi", "eklem sivisi",
        "ενδαρθρικ",
        "αρθρικο υγρο",
        "συλλογη υγρου",
        "ставен излив",
        "излив", "хидропс",
        "zglobni izljev", "izljev",
    ],
    "Synovitis": [
        "synovitis", "synovial thickening", "synovial hypertrophy",
        "thickened synovial", "hypertrophy of the synovium",
        "synovial proliferation", "proliferation of the synovium",
        "sinovitis", "synovite", "synovitiden", "synovialitis",
        "verdikking van het synovium", "verdikkingen van het synovium",
        "synoviale verdikking", "synovialisverdickung", "synovialverdickung",
        "sinovit", "hoffitis", "sinovyal kalinlasma", "sinovyal proliferasyon",
        "συνοβιτ", "υμενιτ",
        "синовит", "sinovij", "sinovije",
    ],
    "Baker's": [
        "baker", "popliteal cyst", "poplitealcyst", "popliteal cysts",
        "quiste popliteo", "quistes popliteos", "quiste de baker",
        "kyste de baker", "kyste poplite", "popliteale cyste", "popliteale cyst",
        "bakercyste", "bakerzyste", "poplitealzyste", "popliteazyste",
        "baker kisti", "popliteal kist",
        "κυστη baker", "κυστη του baker",
        "киста на бейкър",
        "бейкърова киста",
        "poplitealna cista",
    ],
    "Contusion": [
        "contusion", "contusiones", "contusie", "kontusyon", "kontuzyon",
        "bone bruise", "bone bruising", "knochenprellung", "prellung",
        "botcontusie", "μωλωπ", "θλαση",
        "контузи", "kontuzij", "nagnjecen",
    ],
    "Fracture": [
        "fracture", "fractur", "fractura", "fraktur", "fractuur", "breuk",
        "kirik", "kirig", "καταγμα",
        "καταγματ",
        "фрактура", "счупван",
        "avulsion", "avulsie", "avulsiyon", "prijelom",
    ],
}

# Negation cues, word-boundary anchored. A bare "no " substring matches inside the
# Spanish word "cuerno", which silently killed most Spanish positives before this
# was anchored.
NEG_SRC = [
    r"\bno\b", r"\bnot\b", r"\bnon\b", r"\bwithout\b", r"\babsen\w*",
    r"\bnegative for\b", r"\bunremarkable\b", r"\bintact\w*", r"\bnormal\w*",
    r"\bpreserved\b", r"\bfree of\b", r"\bexcluded\b",
    r"\bno hay\b", r"\bsin\b", r"\bausen\w*", r"\bno se\b", r"\bconservad\w*",
    r"\bintegr\w*", r"\bindemne\b",
    r"\bpas de\b", r"\baucun\w*", r"\bsans\b",
    r"\bgeen\b", r"\bzonder\b", r"\bonopvallend\w*", r"\bnormaal\b",
    r"\bvrij van\b", r"\bintacte\b",
    r"\bkein\w*", r"\bohne\b", r"\bunauffallig\w*", r"\bregelrecht\w*",
    r"\bnicht\b", r"\bintakt\w*",
    r"\byok\w*", r"\bizlenmemis\w*", r"\bizlenmedi\w*", r"\bsaptanmamis\w*",
    r"\bgozlenmemis\w*", r"\bgorulmemis\w*", r"\bkorunmus\w*",
    r"\bmevcut degil\b", r"\bnormaldir\b", r"\bdogaldir\b",
    r"\bδεν\b", r"\bχωρις\b",
    r"\bφυσιολογικ\w*",
    r"\bακεραι\w*",
    r"\bбез\b", r"\bняма\b",
    r"\bне се\b", r"\bнормал\w*",
    r"\bзапазен\w*",
    r"\bсъхранен\w*",
    r"\bsenza\b", r"\bsem\b", r"\bnao\b", r"\bbez\b", r"\buredn\w*",
]
NEG_RE = re.compile("|".join(NEG_SRC))

NEG_BACK = 70
NEG_FWD = 45
QUAL_WINDOW = 90
PAIR_WINDOW = 160
CLAUSE_SPLIT = re.compile(r"[.;:\n\r•·]+|\s-\s|\s>\s|\s\*\s")
PF_MASK = re.compile(r"(medial|lateral)\s+(patellar|patella|facet|trochlea|trochlear|retinac)\w*")


def fold_text(text):
    """Lowercase, normalise script-specific letters, and strip diacritics.

    Turkish dotless i, the German sharp s and the Croatian barred d have no
    combining-mark decomposition, so they are mapped explicitly before NFKD.

    Args:
        text: Any report text.

    Returns:
        A lowercase, accent-free string safe for substring matching.
    """
    t = str(text).lower().translate(_CHAR_MAP)
    t = unicodedata.normalize("NFKD", t)
    return "".join(c for c in t if not unicodedata.combining(c))


def _find_any(clause, terms):
    """Return the earliest index at which any term occurs, or -1.

    Args:
        clause: Folded clause text.
        terms: Iterable of folded surface forms.

    Returns:
        Character index of the earliest hit, or -1 when none matches.
    """
    best = -1
    for t in terms:
        i = clause.find(t)
        if i >= 0 and (best < 0 or i < best):
            best = i
    return best


def _negated(clause, pos, span):
    """Test whether a negation cue sits near a matched finding term.

    Args:
        clause: Folded clause text.
        pos: Start index of the finding term.
        span: Length of the finding term.

    Returns:
        True when a cue appears in the preceding or following window.
    """
    back = clause[max(0, pos - NEG_BACK):pos]
    fwd = clause[pos + span:pos + span + NEG_FWD]
    return bool(NEG_RE.search(back) or NEG_RE.search(fwd))


def _fire_direct(clause, terms):
    """Test a direct finding term, retrying later occurrences past a negation.

    Args:
        clause: Folded clause text.
        terms: Surface forms that are themselves the finding.

    Returns:
        True when at least one occurrence is unnegated.
    """
    for t in terms:
        start = 0
        while True:
            i = clause.find(t, start)
            if i < 0:
                break
            if not _negated(clause, i, len(t)):
                return True
            start = i + 1
    return False


def _anatomy_index(clause, label):
    """Locate the anatomy mention for a paired label.

    Falls back to a generic organ term paired with a side qualifier, which is what
    catches Greek and Bulgarian phrasing that names the compartment rather than the
    structure.

    Args:
        clause: Folded clause text.
        label: One of the four paired label names.

    Returns:
        Character index of the anatomy mention, or -1.
    """
    i = _find_any(clause, SPECIFIC[label])
    if i >= 0:
        return i
    g = _find_any(clause, GENERIC[label])
    if g >= 0 and _find_any(clause, SIDE_Q[label]) >= 0:
        return g
    return -1


def extract_labels(report):
    """Extract twelve binary findings from one free-text radiology report.

    Args:
        report: Report text in any of the languages present in the corpus.

    Returns:
        Dict mapping each of the twelve label names to 0 or 1.
    """
    out = {lab: 0 for lab in LABELS}
    if not isinstance(report, str) or not report.strip():
        return out
    text = fold_text(report)
    for raw in CLAUSE_SPLIT.split(text):
        clause = raw.strip()
        if len(clause) < 3:
            continue
        for lab in ("Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"):
            if not out[lab] and _fire_direct(clause, DIRECT[lab]):
                out[lab] = 1
        for lab in SPECIFIC:
            if out[lab]:
                continue
            ai = _anatomy_index(clause, lab)
            if ai < 0:
                continue
            finds = TEAR + SPRAIN if lab in ("ACL", "MCL") else TEAR
            for t in finds:
                fi = clause.find(t)
                if fi < 0 or abs(fi - ai) > PAIR_WINDOW:
                    continue
                if not _negated(clause, fi, len(t)):
                    out[lab] = 1
                    break
        oi = _find_any(clause, OA_FIND)
        if oi >= 0 and not _negated(clause, oi, 8):
            if _find_any(clause, TRICOMP) >= 0:
                out["Medial OA"] = 1
                out["Lateral OA"] = 1
                out["PF OA"] = 1
            win = clause[max(0, oi - QUAL_WINDOW):oi + QUAL_WINDOW]
            if _find_any(win, PF_Q) >= 0:
                out["PF OA"] = 1
            masked = PF_MASK.sub(" ", win)
            if _find_any(masked, MEDIAL_Q) >= 0:
                out["Medial OA"] = 1
            if _find_any(masked, LATERAL_Q) >= 0:
                out["Lateral OA"] = 1
    return out


print("extractor ready:", len(SPECIFIC), "paired labels,",
      len(DIRECT), "direct labels,", len(NEG_SRC), "negation cues")

In [ ]:
def _label_search_dirs():
    """Directories that may hold a pre-extracted label table.

    Kaggle nests every input under a category directory rather than mounting it flat.
    The competition lands at /kaggle/input/competitions/<slug>, the model at
    /kaggle/input/models/<owner>/<name>/<framework>/<variation>/<version>, and an
    attached dataset at /kaggle/input/datasets/<owner>/<slug>. Version 2 of this
    notebook looked only at the direct children of /kaggle/input, found no label file,
    and fell back to the regex extractor while reporting a clean run, which measured
    the exact artifact the LLM extraction was built to beat.

    So this walks a bounded depth instead of listing one level, and prunes the
    competition tree, which holds roughly 730,000 DICOM files and would otherwise make
    the walk cost more than the encoding.

    Returns:
        List of Path objects, existing directories only, shallowest first.
    """
    out = []
    kin = Path("/kaggle/input")
    if kin.is_dir():
        out.append(kin)
        stack = [(kin, 0)]
        while stack:
            d, depth = stack.pop()
            if depth >= 5:
                continue
            try:
                children = sorted(p for p in d.iterdir() if p.is_dir())
            except OSError:
                continue
            for c in children:
                # Never descend into the DICOM corpus or a git/cache tree.
                if c.name in ("competitions", "train_series", "test_series",
                              ".git", "__pycache__"):
                    continue
                out.append(c)
                stack.append((c, depth + 1))
    out.append(Path("/mnt/data/Github/Kaggle/Competitions"
                    "/rsna_knee_abnormality_detection/data/interim"))
    out.append(Path("data/interim"))
    seen, uniq = set(), []
    for d in out:
        if str(d) not in seen and d.is_dir():
            seen.add(str(d))
            uniq.append(d)
    return uniq


def _find_label_file(patterns):
    """Find the newest file matching any of the given glob patterns.

    Args:
        patterns: Glob patterns, tried in order across every search directory.

    Returns:
        A Path, or None when nothing matches.
    """
    hits = []
    for d in _label_search_dirs():
        for pat in patterns:
            try:
                hits.extend(sorted(d.glob(pat)))
            except OSError:
                continue
    if not hits:
        return None
    return sorted(hits, key=lambda p: (p.name, str(p)))[-1]


def _read_label_table(path):
    """Read a label table and keep only the id column and the twelve findings.

    Args:
        path: Parquet or CSV path.

    Returns:
        DataFrame indexed by StudyInstanceUID with the twelve label columns as
        floats, or None when the file cannot be read or lacks the columns.
    """
    try:
        df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    except Exception as exc:
        print("could not read", path, ":", exc)
        return None
    if "StudyInstanceUID" not in df.columns or not set(LABELS).issubset(df.columns):
        print("label file", path.name, "is missing required columns; ignored")
        return None
    out = df[["StudyInstanceUID"] + LABELS].copy()
    out["StudyInstanceUID"] = out["StudyInstanceUID"].astype(str)
    out = out.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")
    return out[LABELS].astype(float).clip(0.0, 1.0)


train_ids = train["StudyInstanceUID"].astype(str).tolist()
report_col = "Report" if "Report" in train.columns else None

# ---- 1. Always compute the rule labels; they are the floor and the fallback ----
rule_df = None
if LABEL_SOURCE in ("auto", "rule_file"):
    p = _find_label_file(RULE_LABEL_PATTERNS)
    if p is not None:
        rule_df = _read_label_table(p)
        if rule_df is not None:
            print("rule labels from file :", p.name, "|", len(rule_df), "studies")
if rule_df is None:
    if report_col is None:
        raise KeyError("train.csv has no Report column and no label file was found")
    t0 = time.time()
    rule_df = pd.DataFrame([extract_labels(r) for r in train[report_col]],
                           index=pd.Index(train_ids, name="StudyInstanceUID"))
    rule_df = rule_df[LABELS].astype(float)
    print("rule labels extracted inline: {} reports in {:.1f}s".format(
        len(rule_df), time.time() - t0))

Y_soft = rule_df.reindex(train_ids).astype(float)
Y_soft = Y_soft.fillna(0.0)
# Confidence is target-specific.  Unknown report cells in Pilkwang's table
# carry confidence 0.05 and must not count like explicit positive/negative text.
Y_CONF = pd.DataFrame(RULE_WEIGHT, index=train_ids, columns=LABELS, dtype=float)
LABEL_PROVENANCE = "rule"
LLM_LABEL_PATH = None
n_llm = 0

# ---- 2. Overwrite with LLM soft labels wherever they exist --------------------
if LABEL_SOURCE in ("auto", "llm"):
    p = _find_label_file(LLM_LABEL_PATTERNS)
    if p is not None:
        llm = _read_label_table(p)
        if llm is not None:
            common = [s for s in train_ids if s in llm.index]
            if common:
                Y_soft.loc[common, LABELS] = llm.loc[common, LABELS].values
                n_llm = len(common)
                LLM_LABEL_PATH = p
                label_sha = hashlib.sha256(p.read_bytes()).hexdigest()
                if label_sha != EXPECTED_LABEL_SHA256:
                    raise RuntimeError('label table SHA-256 {} != pinned {}'.format(
                        label_sha, EXPECTED_LABEL_SHA256))
                raw_conf = (pd.read_parquet(p) if p.suffix == '.parquet'
                            else pd.read_csv(p))
                raw_conf['StudyInstanceUID'] = raw_conf['StudyInstanceUID'].astype(str)
                raw_conf = raw_conf.drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID')
                for target in LABELS:
                    col = target + '__conf'
                    if col in raw_conf.columns:
                        Y_CONF.loc[common, target] = (
                            pd.to_numeric(raw_conf.loc[common, col], errors='coerce')
                            .fillna(RULE_WEIGHT).clip(0.01, 1.0).values
                        )
                LABEL_PROVENANCE = "llm+rule" if n_llm < len(train_ids) else "llm"
                print("LLM labels from file  :", p.name, "|", n_llm,
                      "of", len(train_ids), "studies overwritten")
    elif LABEL_SOURCE == "llm":
        print("LABEL_SOURCE='llm' but no LLM label file is mounted; using rule labels")

# ---- 2b. Fail closed when a declared label dataset did not arrive -------------
# Falling back to the regex extractor is the right behaviour when no label dataset was
# ever declared. It is the WRONG behaviour when one was declared and did not mount:
# the run then measures the artifact the extraction exists to beat, and reports a clean
# result while doing it. Version 2 of this notebook did exactly that. A guard that
# treats "the good input is missing" as permission to continue is fail-open.
if LABEL_DATASET_SOURCES and LABEL_PROVENANCE not in ("llm", "llm+rule"):
    _listing = []
    _kin = Path("/kaggle/input")
    if _kin.is_dir():
        for _d in _label_search_dirs()[:40]:
            try:
                _listing.append(f"{_d} -> "
                                + ", ".join(sorted(p.name for p in _d.iterdir())[:6]))
            except OSError:
                continue
    raise RuntimeError(
        "Declared label dataset(s) " + ", ".join(LABEL_DATASET_SOURCES)
        + " but no file matched " + " / ".join(LLM_LABEL_PATTERNS)
        + f", so labels resolved to '{LABEL_PROVENANCE}'. Refusing to train on the "
          "fallback labels, because the whole point of this run is the extracted ones. "
          "Directories searched:\n  " + "\n  ".join(_listing)
    )

# ---- 3. Gold labels are ground truth and override anything extracted ----------
if N_GOLD:
    gold_vals = (train.loc[gold_mask, ["StudyInstanceUID"] + LABELS]
                 .set_index(train.loc[gold_mask, "StudyInstanceUID"].astype(str)))
    Y_soft.loc[gold_ids, LABELS] = gold_vals[LABELS].astype(float).values
    Y_CONF.loc[gold_ids, LABELS] = GOLD_WEIGHT

IS_GOLD = pd.Series([s in set(gold_ids) for s in train_ids], index=train_ids)
SAMPLE_WEIGHT = Y_CONF.mean(axis=1).astype(np.float32).values

prev = pd.DataFrame({
    "mean_soft_target": Y_soft.mean().round(4),
    "positives_at_0.5": (Y_soft >= 0.5).sum().astype(int),
    "prevalence_%": ((Y_soft >= 0.5).mean() * 100).round(1),
})
print()
print("label source      :", LABEL_SOURCE, "-> resolved to", LABEL_PROVENANCE)
print("gold overrides    :", N_GOLD, "studies, weighted x{:.0f} in the loss".format(
    GOLD_WEIGHT))
print()
print(prev.to_string())
print()
print('target confidence : mean {:.3f} | p10 {:.3f} | p90 {:.3f}'.format(
    float(Y_CONF.values.mean()), float(np.quantile(Y_CONF.values, 0.10)),
    float(np.quantile(Y_CONF.values, 0.90))))

In [ ]:
PLANES = ["Sagittal", "Coronal", "Axial"]


def normalise_manifest(df):
    """Coerce a series manifest into the columns the slot builder needs.

    Args:
        df: A raw train_series or test_series frame.

    Returns:
        A copy with string study/series ids, a title-cased plane and an integer
        fluid flag. Empty in, empty out.
    """
    out = df.copy()
    for c in SERIES_COLS:
        if c not in out.columns:
            out[c] = np.nan
    out["StudyInstanceUID"] = out["StudyInstanceUID"].astype(str)
    out["SeriesInstanceUID"] = out["SeriesInstanceUID"].astype(str)
    out["plane"] = out["Anatomical_Plane"].astype(str).str.strip().str.title()
    out["fluid"] = pd.to_numeric(out["Fluid_Sensitive"], errors="coerce") \
        .fillna(0).astype(int)
    return out


tr_man = normalise_manifest(train_series)
te_man = normalise_manifest(test_series)

# The flag check comes first, because the whole six-slot design downstream rests on
# it. Fluid_Sensitive and Fat_Suppression are one column wearing two names.
if len(tr_man) and "Fat_Suppression" in train_series.columns:
    fs = pd.to_numeric(train_series["Fluid_Sensitive"], errors="coerce")
    fat = pd.to_numeric(train_series["Fat_Suppression"], errors="coerce")
    n_disagree = int((fs != fat).sum())
    print("Fluid_Sensitive vs Fat_Suppression over {:,} series: {} rows disagree"
          .format(len(train_series), n_disagree))
    print("  -> {} contrast classes, not four".format(2 if n_disagree == 0 else 4))
    print("  fluid-sensitive series {:,} | non-fluid {:,}".format(
        int((fs == 1).sum()), int((fs == 0).sum())))
    print()


def slot_fill_table(man):
    """Measure how often each plane-by-contrast slot has a genuine series.

    Args:
        man: A normalised manifest.

    Returns:
        DataFrame with one row per candidate slot, sorted by fill rate.
    """
    n = man["StudyInstanceUID"].nunique()
    rows = []
    for plane in PLANES:
        for f in (1, 0):
            sub = man[(man["plane"] == plane) & (man["fluid"] == f)]
            have = sub["StudyInstanceUID"].nunique()
            rows.append({
                "slot": "{}_{}".format(plane, "Fluid" if f else "Structural"),
                "series": len(sub),
                "studies": have,
                "fill": have / max(1, n),
                "kept": "{}_{}".format(plane, "Fluid" if f else "Structural")
                        in SLOT_NAMES,
            })
    return pd.DataFrame(rows).sort_values("fill", ascending=False)


fill = slot_fill_table(tr_man)
print("Six candidate slots, measured over {:,} training studies".format(
    tr_man["StudyInstanceUID"].nunique()))
print(fill.to_string(index=False, float_format=lambda v: "{:.4f}".format(v)))
print()

# The frontier's rule fills a plane's second slot whenever the plane holds two or
# more series, whatever contrast they are. Reproduce that so the comparison is a
# measurement rather than a claim.
cnt = (tr_man.groupby(["StudyInstanceUID", "plane"]).size().unstack(fill_value=0)
       .reindex(columns=PLANES, fill_value=0))
n_studies_tr = max(1, len(cnt))
rival_fill = 3.0 + sum((cnt[p] >= 2).mean() for p in PLANES)
print("frontier's six-slot rule, second slot per plane fills when the plane has >=2 series:")
for p in PLANES:
    print("  {:9s} {:5.1f}% of studies".format(p, 100.0 * (cnt[p] >= 2).mean()))
ax_two = set(cnt.index[cnt["Axial"] >= 2])
ax_true = set(tr_man.loc[(tr_man["plane"] == "Axial") & (tr_man["fluid"] == 0),
                         "StudyInstanceUID"])
print("  mean slots filled {:.3f} / 6 = {:.1%} per slot".format(
    rival_fill, rival_fill / 6.0))
print("  of the {} studies whose Axial second slot fills, {} have no genuine non-fluid"
      " axial series, so the slot holds a duplicate contrast".format(
          len(ax_two), len(ax_two - ax_true)))
print()

kept = fill[fill["kept"]]
mean_fill = kept["fill"].sum()
print("this notebook's {} slots, contrast enforced:".format(N_SLOTS))
for name, (_, _, k) in zip(SLOT_NAMES, SLOT_SPEC):
    row = fill.loc[fill["slot"] == name].iloc[0]
    print("  {:22s} fill {:6.1%}  slice budget {:2d}".format(name, row["fill"], k))
print("  mean slots filled {:.3f} / {} = {:.1%} per slot".format(
    mean_fill, N_SLOTS, mean_fill / N_SLOTS))
exp_slices = sum(fill.loc[fill["slot"] == n, "fill"].iloc[0] * k
                 for n, (_, _, k) in zip(SLOT_NAMES, SLOT_SPEC))
print("  expected slices encoded per study: {:.1f}".format(exp_slices))
print()


def build_slots(man):
    """Assign one series to each contrast-defined slot, per study.

    A slot is filled only by a series whose plane and fluid flag both match. A
    slot with no matching series stays empty and is carried as a masked, zero
    embedding rather than being back-filled with the wrong contrast.

    Args:
        man: A normalised manifest.

    Returns:
        Dict mapping study id to a dict of slot name -> series id.
    """
    out = {}
    if not len(man):
        return out
    for sid, rows in man.groupby("StudyInstanceUID", sort=False):
        chosen = {}
        used = set()
        for name, (plane, flag, _) in zip(SLOT_NAMES, SLOT_SPEC):
            part = rows[(rows["plane"] == plane) & (rows["fluid"] == flag)]
            for series_id in part["SeriesInstanceUID"].tolist():
                if series_id not in used:
                    chosen[name] = series_id
                    used.add(series_id)
                    break
        out[str(sid)] = chosen
    return out


SLOTS_TRAIN = build_slots(tr_man)
SLOTS_TEST = build_slots(te_man)
filled_tr = np.array([len(SLOTS_TRAIN.get(s, {})) for s in train_ids])
print("slot assignment    : train {} studies, mean {:.3f} slots filled".format(
    len(SLOTS_TRAIN), filled_tr.mean() if len(filled_tr) else 0.0))
print("                     test  {} studies".format(len(SLOTS_TEST)))

In [ ]:
GEOM_TAGS = ["ImagePositionPatient", "ImageOrientationPatient",
             "SliceLocation", "InstanceNumber", "Laterality", "ImageLaterality"]

# Counters, so a degraded run is visible in the log rather than silent. They are
# incremented from worker threads, and `d[k] += 1` is three bytecodes, not one, so
# the lock is not decoration: an undercounted failure reads as a clean run.
import threading

STATS = {"series_missing": 0, "header_fail": 0, "pixel_fail": 0,
         "series_truncated": 0, "slots_empty": 0, "slices_decoded": 0,
         "exact_order": 0, "fallback_order": 0,
         "lat_left": 0, "lat_right_mirrored": 0, "lat_right_reversed": 0,
         "lat_absent": 0, "lat_conflict": 0}
_STATS_LOCK = threading.Lock()


def bump(key, n=1):
    """Increment one diagnostic counter under a lock.

    Args:
        key: Counter name.
        n: Amount to add.

    Returns:
        None.
    """
    with _STATS_LOCK:
        STATS[key] = STATS.get(key, 0) + n


def list_dicom_paths(folder):
    """List the .dcm files in one series directory.

    Args:
        folder: Series directory.

    Returns:
        A list of Paths sorted by filename, empty when the directory is missing
        or unreadable. Filename order is SOP UID order here, which is arbitrary;
        it is used only to take a deterministic subset, never as slice order.
    """
    try:
        return sorted((Path(e.path) for e in os.scandir(folder)
                       if e.is_file() and e.name.lower().endswith(".dcm")),
                      key=lambda p: p.name)
    except Exception:
        return []


def projected_position(ds, fallback):
    """Signed position of a slice along its own stack normal, in millimetres.

    The two in-plane direction cosines in ImageOrientationPatient span the slice
    plane; their cross product is the slice normal. Projecting
    ImagePositionPatient onto that normal gives a scalar that increases
    monotonically through the stack and is invariant to how the scanner numbered
    the instances. SliceLocation and InstanceNumber are fallbacks only.

    Args:
        ds: A pydicom dataset read with stop_before_pixels.
        fallback: Value to return when no geometry tag is usable.

    Returns:
        Float position.
    """
    try:
        iop = np.asarray(ds.ImageOrientationPatient, dtype=np.float64)
        ipp = np.asarray(ds.ImagePositionPatient, dtype=np.float64)
        if iop.size == 6 and ipp.size == 3:
            normal = np.cross(iop[:3], iop[3:])
            norm = float(np.linalg.norm(normal))
            if norm > 1e-8:
                return float(np.dot(ipp, normal / norm))
    except Exception:
        pass
    for attr in ("SliceLocation", "InstanceNumber"):
        try:
            v = float(getattr(ds, attr))
            if np.isfinite(v):
                return v
        except Exception:
            pass
    return float(fallback)


def read_laterality(ds):
    """Read which knee this slice belongs to, as "L", "R" or "" for unknown.

    Two tags carry it and neither is mandatory. `Laterality` is the general one;
    `ImageLaterality` is what some vendors write instead. The corpus contains at
    least the spellings L, R and LEFT, so the value is folded to its first
    character rather than compared whole, and anything that is not L or R is
    treated as unknown rather than guessed at.

    Args:
        ds: A pydicom dataset read with the GEOM_TAGS allowlist.

    Returns:
        "L", "R", or "" when neither tag is present or the value is unrecognised.
    """
    for tag in ("Laterality", "ImageLaterality"):
        v = getattr(ds, tag, None)
        if v is None:
            continue
        s = str(v).strip().upper()
        if s[:1] in ("L", "R"):
            return s[:1]
    return ""


def resolve_laterality(votes):
    """Collapse per-slice laterality readings into one value for the series.

    Args:
        votes: Iterable of per-slice values from `read_laterality`.

    Returns:
        "L", "R", or "" when the series is unlabelled or self-contradictory. A
        series that claims both sides is not a series whose geometry is
        understood, so it is left alone rather than resolved by majority.
    """
    seen = {v for v in votes if v}
    if len(seen) == 1:
        return seen.pop()
    if len(seen) > 1:
        bump("lat_conflict")
    return ""


def order_series(folder):
    """Order one series' slices by projected physical position.

    Headers are read with stop_before_pixels and a four-tag allowlist, so no
    pixel decoder is involved and none of the four transfer syntaxes in this
    corpus can fail the read. Series longer than the cap are subsampled by
    filename first, which is a uniform draw over positions because filenames are
    SOP UIDs.

    Args:
        folder: Series directory.

    Returns:
        Tuple of (ordered paths, ordered positions, number of files on disk,
        whether every file on disk was ordered, series laterality as "L", "R"
        or "").
    """
    paths = list_dicom_paths(folder)
    n_files = len(paths)
    if n_files == 0:
        bump("series_missing")
        return [], np.zeros(0), 0, False, ""

    exact = n_files <= MAX_HEADER_READS_PER_SERIES
    if not exact:
        bump("series_truncated")
        idx = np.linspace(0, n_files - 1, MAX_HEADER_READS_PER_SERIES)
        idx = np.unique(np.round(idx).astype(int))
        paths = [paths[i] for i in idx]

    positioned, lat_votes = [], []
    for i, p in enumerate(paths):
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True,
                                 specific_tags=GEOM_TAGS, force=True)
            positioned.append((projected_position(ds, i), p))
            lat_votes.append(read_laterality(ds))
        except Exception:
            bump("header_fail")
            positioned.append((float(i), p))
    positioned.sort(key=lambda t: t[0])
    bump("exact_order" if exact else "fallback_order")
    return ([t[1] for t in positioned],
            np.asarray([t[0] for t in positioned], dtype=np.float64),
            n_files, exact, resolve_laterality(lat_votes))


def pick_indices(n, k, lo, hi):
    """Choose k evenly spaced indices across a fractional window of n slices.

    Args:
        n: Number of ordered slices available.
        k: How many to take.
        lo: Lower fractional bound of the window, in [0, 1].
        hi: Upper fractional bound.

    Returns:
        A sorted array of unique integer indices, shorter than k when n is small.
    """
    if n <= 0 or k <= 0:
        return np.zeros(0, dtype=int)
    if n <= k:
        return np.arange(n)
    fracs = np.linspace(lo, hi, k)
    idx = np.round(fracs * (n - 1)).astype(int)
    return np.unique(np.clip(idx, 0, n - 1))

In [ ]:
def read_slice(path):
    """Decode one DICOM slice into a rescaled float array.

    RescaleSlope and RescaleIntercept are applied before anything else, and
    MONOCHROME1 is inverted so bright always means high signal. No window is
    applied here: the window belongs to the series, not the slice.

    Args:
        path: Path to a .dcm file.

    Returns:
        A 2D float32 array, or None when the file cannot be decoded.
    """
    try:
        ds = pydicom.dcmread(str(path), force=True)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        bump("pixel_fail")
        return None
    try:
        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        arr = arr * slope + intercept
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            arr = float(np.nanmax(arr)) - arr
    except Exception:
        pass
    if not np.isfinite(arr).any():
        bump("pixel_fail")
        return None
    bump("slices_decoded")
    return arr


def series_window(arrays, lo_pct=1.0, hi_pct=99.5):
    """Outlier-resistant intensity window for a series, from its pooled pixels.

    This is the correction that matters most in preprocessing. A per-slice
    window rescales a slice through a large effusion and a slice through dry
    bone onto the same range, which deletes the very contrast that Effusion,
    Synovitis and Baker's are made of, and gives the three channels of one 2.5D
    image three different windows. One window per series keeps relative
    brightness intact within the series while still absorbing the sharp
    between-series intensity differences the EDA found.

    Args:
        arrays: List of 2D float arrays from one series.
        lo_pct: Lower percentile.
        hi_pct: Upper percentile.

    Returns:
        Tuple (low, high) with high strictly greater than low.
    """
    pool = []
    for a in arrays:
        flat = a[::3, ::3].ravel()
        flat = flat[np.isfinite(flat)]
        if flat.size:
            pool.append(flat)
    if not pool:
        return 0.0, 1.0
    pool = np.concatenate(pool)
    nz = pool[np.abs(pool) > 1e-8]
    src = nz if nz.size >= 256 else pool
    lo, hi = np.percentile(src, [lo_pct, hi_pct])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(pool)), float(np.max(pool))
    if hi <= lo:
        hi = lo + 1.0
    return float(lo), float(hi)


def series_bbox(arrays, low, high, thresh=0.06, margin=0.06):
    """Body bounding box for a whole series, from the union of its slices.

    Computed once per series so the crop does not jitter between neighbouring
    slices, which would put the same anatomy in different patch positions in the
    three channels of one image.

    Args:
        arrays: List of 2D float arrays from one series.
        low: Series window low.
        high: Series window high.
        thresh: Normalised intensity above which a pixel counts as tissue.
        margin: Fractional margin added around the union box.

    Returns:
        Tuple (y0, y1, x0, x1). The full frame when no box can be found.
    """
    h, w = arrays[0].shape
    y0, y1, x0, x1 = h, 0, w, 0
    span = max(high - low, 1e-6)
    for a in arrays:
        if a.shape != (h, w):
            return 0, h, 0, w
        m = ((a - low) / span) > thresh
        if m.sum() < 0.01 * h * w:
            continue
        ys, xs = np.where(m)
        y0 = min(y0, int(ys.min())); y1 = max(y1, int(ys.max()) + 1)
        x0 = min(x0, int(xs.min())); x1 = max(x1, int(xs.max()) + 1)
    if y1 <= y0 or x1 <= x0:
        return 0, h, 0, w
    my = max(2, int(margin * (y1 - y0)))
    mx = max(2, int(margin * (x1 - x0)))
    return (max(0, y0 - my), min(h, y1 + my), max(0, x0 - mx), min(w, x1 + mx))


CROP_RATIOS = []


def normalise_laterality(planes, plane, lat):
    """Map a right knee onto the left-knee convention.

    Four of the twelve targets are medial/lateral pairs, and medial and lateral
    are defined against the body midline rather than against the image. On a
    right knee they therefore fall on the opposite side of the frame from a left
    knee, so without this the four side-specific labels are being asked to learn
    from an axis the model cannot observe.

    The correction differs by plane because the mirror acts on a different axis:

    * Coronal and axial carry the medial-lateral direction in the image plane, so
      one knee is the horizontal mirror of the other and the last axis flips.
    * Sagittal carries it along the *slice* axis. Each slice is unchanged by the
      mirror; what differs is the direction in which the stack traverses the
      joint, so the stack order reverses and the pixels are left alone.

    A study with no `Laterality` tag is left untouched. Roughly half the corpus
    is unlabelled, and a wrong flip is worse than no flip: it moves a study to
    the wrong side rather than leaving it on an unknown one.

    Args:
        planes: List of normalised 2D slices in through-plane order.
        plane: Anatomical plane of this slot.
        lat: Series laterality, "L", "R" or "".

    Returns:
        Tuple of (possibly transformed planes, whether the stack was reversed).
    """
    if lat != "R":
        bump("lat_left" if lat == "L" else "lat_absent")
        return planes, False
    if plane in ("Coronal", "Axial"):
        bump("lat_right_mirrored")
        return [p[:, ::-1] for p in planes], False
    bump("lat_right_reversed")
    return planes[::-1], True


def build_slot_stack(folder, plane, k, size):
    """Decode, window, crop and resize one slot into 2.5D image channels.

    Every encoded image is the decoded slice plus its two decoded neighbours as
    the three channels, so neighbourhood context costs no extra DICOM reads:
    each decoded slice is reused in up to three images.

    Args:
        folder: Series directory.
        plane: Anatomical plane, selecting the sampling window.
        k: How many slices to encode from this slot.
        size: Output side length in pixels.

    Returns:
        Tuple of (images uint8 [m, 3, size, size], positions float32 [m],
        millimetres spanned, number of files on disk). m may be 0.
    """
    paths, positions, n_files, _, lat = order_series(folder)
    if not paths:
        return np.zeros((0, 3, size, size), np.uint8), np.zeros(0, np.float32), 0.0, 0
    lo, hi = PLANE_WINDOW.get(plane, (0.15, 0.85))
    idx = pick_indices(len(paths), k, lo, hi)
    if idx.size == 0:
        return np.zeros((0, 3, size, size), np.uint8), np.zeros(0, np.float32), 0.0, n_files

    arrays, kept = [], []
    for i in idx:
        a = read_slice(paths[int(i)])
        if a is not None:
            arrays.append(a)
            kept.append(int(i))
    if not arrays:
        return np.zeros((0, 3, size, size), np.uint8), np.zeros(0, np.float32), 0.0, n_files

    low, high = series_window(arrays)
    y0, y1, x0, x1 = series_bbox(arrays, low, high)
    h, w = arrays[0].shape
    if h * w > 0:
        CROP_RATIOS.append(math.sqrt(max(1, (y1 - y0) * (x1 - x0)) / float(h * w)))

    planes_norm = []
    span = max(high - low, 1e-6)
    for a in arrays:
        if a.shape != (h, w):
            a = resize_2d(a, max(h, w))[:h, :w]
        c = a[y0:y1, x0:x1]
        c = np.clip((c - low) / span, 0.0, 1.0)
        side = max(c.shape)
        pad = np.zeros((side, side), np.float32)
        oy, ox = (side - c.shape[0]) // 2, (side - c.shape[1]) // 2
        pad[oy:oy + c.shape[0], ox:ox + c.shape[1]] = c
        planes_norm.append(np.clip(resize_2d(pad, size) * 255.0, 0, 255).astype(np.uint8))

    # Section 6b. Applied before the 2.5D channels are assembled, so a reversed
    # sagittal stack still pairs each slice with its true neighbours rather than
    # with the neighbours it had in the un-normalised direction.
    planes_norm, stack_reversed = normalise_laterality(planes_norm, plane, lat)

    m = len(planes_norm)
    imgs = np.zeros((m, 3, size, size), np.uint8)
    for j in range(m):
        imgs[j, 0] = planes_norm[max(0, j - 1)]
        imgs[j, 1] = planes_norm[j]
        imgs[j, 2] = planes_norm[min(m - 1, j + 1)]

    pos = positions[np.asarray(kept, dtype=int)] if positions.size else np.arange(m)
    span_mm = float(np.abs(positions.max() - positions.min())) if positions.size > 1 else 0.0
    rng = max(1e-6, float(pos.max() - pos.min())) if m > 1 else 1.0
    frac = ((pos - pos.min()) / rng).astype(np.float32) if m > 1 else np.zeros(m, np.float32)
    if stack_reversed:
        # The head reads `frac` as distance along the slot's own axis. Reversing
        # the images without reversing this would hand every reversed study a
        # positional encoding that runs backwards against its own pixels.
        frac = (1.0 - frac[::-1]).astype(np.float32)
    return imgs, frac, span_mm, n_files


print("preprocessing ready")
print("laterality                   : right knees mapped onto the left convention, "
      "coronal/axial mirrored, sagittal stack reversed, untagged left alone")
print("per-series window percentiles: 1.0 / 99.5, pooled over the slices encoded")
print("sampling window by plane      :",
      ", ".join("{} {:.2f}-{:.2f}".format(k, v[0], v[1])
                for k, v in PLANE_WINDOW.items()))

In [ ]:
SLOT_ORDER = list(zip(SLOT_NAMES, SLOT_SPEC))


def study_stacks(study_id, dcm_root, slots, size):
    """Decode the two selected series in parallel into ordered 2.5D images."""
    chosen = slots.get(str(study_id), {})

    def one(args):
        i, (name, (plane, _flag, k)) = args
        series_id = chosen.get(name)
        if series_id is None or dcm_root is None:
            bump("slots_empty")
            return i, (np.zeros((0, 3, size, size), np.uint8),
                       np.zeros(0, np.float32), 0.0, 0)
        folder = Path(dcm_root) / str(study_id) / str(series_id)
        try:
            return i, build_slot_stack(folder, plane, k, size)
        except Exception:
            bump("slots_empty")
            return i, (np.zeros((0, 3, size, size), np.uint8),
                       np.zeros(0, np.float32), 0.0, 0)

    with ThreadPoolExecutor(max_workers=min(IO_WORKERS, N_SLOTS)) as pool:
        return dict(pool.map(one, list(enumerate(SLOT_ORDER))))


test_ids = test["StudyInstanceUID"].astype(str).tolist()
rest = [s for s in train_ids if s not in set(gold_ids)]
_rng = np.random.default_rng(SEED)
_rng.shuffle(rest)
TRAIN_ENCODE_IDS = gold_ids + rest
print("cache order: {} train (gold first), {} placeholder test".format(
    len(TRAIN_ENCODE_IDS), len(test_ids)))


In [ ]:
# Deterministic orthogonal channel projection, shared by train and test.
_proj_rng = np.random.default_rng(SEED)
_proj_np = _proj_rng.standard_normal((EMBED_DIM, REGION_DIM)).astype(np.float32)
_proj_np, _ = np.linalg.qr(_proj_np)
REGION_PROJ = torch.from_numpy(_proj_np[:, :REGION_DIM].copy()).to(DEVICE)


@torch.no_grad()
def embed_spatial_regions(imgs):
    """Return 8x8 region tokens with spatial layout intact."""
    if DINO is None or len(imgs) == 0:
        return np.zeros((len(imgs), REGION_GRID, REGION_GRID, REGION_DIM), np.float16)
    out = np.zeros((len(imgs), REGION_GRID, REGION_GRID, REGION_DIM), np.float16)
    mean = torch.from_numpy(IMAGENET_MEAN).to(DEVICE)
    std = torch.from_numpy(IMAGENET_STD).to(DEVICE)
    for start in range(0, len(imgs), GPU_BATCH):
        chunk = imgs[start:start + GPU_BATCH]
        x = torch.from_numpy(chunk).to(DEVICE).float().div_(255.0)
        x = (x - mean) / std
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(DEVICE == "cuda")):
            hidden = DINO(pixel_values=x, **DINO_KW).last_hidden_state[:, 1:]
        side = int(round(math.sqrt(hidden.shape[1])))
        if side * side != hidden.shape[1]:
            raise RuntimeError("DINO patch token count is not square")
        grid = hidden.float().transpose(1, 2).reshape(len(chunk), EMBED_DIM, side, side)
        pooled = torch.nn.functional.adaptive_avg_pool2d(
            grid, (REGION_GRID, REGION_GRID)
        ).permute(0, 2, 3, 1)
        projected = pooled @ REGION_PROJ
        out[start:start + len(chunk)] = projected.cpu().numpy().astype(np.float16)
    return out


def cache_paths(split):
    stem = "e005_{}_{}".format(split, SPATIAL_CACHE_VERSION)
    return {
        "emb": WORK / (stem + ".npy"),
        "mask": WORK / (stem + "_mask.npy"),
        "pos": WORK / (stem + "_pos.npy"),
        "ids": WORK / (stem + "_ids.csv"),
    }


def encode_spatial(ids, dcm_root, slots, split, budget_s):
    f = cache_paths(split)
    shape = (len(ids), N_SLOTS, MAX_SLICES, REGIONS, REGION_DIM)
    projected_bytes = int(np.prod(shape)) * np.dtype(np.float16).itemsize
    if projected_bytes + 2_000_000_000 > shutil.disk_usage(WORK).free:
        raise RuntimeError("insufficient Kaggle working disk for spatial cache")
    emb = np.lib.format.open_memmap(f["emb"], mode="w+", dtype=np.float16, shape=shape)
    mask = np.zeros((len(ids), N_SLOTS, MAX_SLICES), np.bool_)
    pos = np.zeros((len(ids), N_SLOTS, MAX_SLICES), np.float16)
    deadline = time.time() + budget_s
    done = 0
    report_every = max(1, len(ids) // 20)
    t0 = time.time()
    for i, sid in enumerate(ids):
        if time.time() > deadline:
            print("{} cache budget exhausted after {} of {}".format(split, i, len(ids)))
            break
        stacks = study_stacks(sid, dcm_root, slots, IMG_SIZE)
        for slot_i, (imgs, frac, _span, _nfiles) in stacks.items():
            m = min(len(imgs), MAX_SLICES)
            if m == 0:
                continue
            emb[i, slot_i, :m] = embed_spatial_regions(imgs[:m]).reshape(
                m, REGIONS, REGION_DIM
            )
            mask[i, slot_i, :m] = True
            pos[i, slot_i, :m] = frac[:m].astype(np.float16)
        if mask[i].any():
            done += 1
        if (i + 1) % report_every == 0:
            rate = (i + 1) / max(time.time() - t0, 1e-6)
            print("  {} {:5d}/{:5d} {:.2f} studies/s eta {:.0f}s".format(
                split, i + 1, len(ids), rate, (len(ids) - i - 1) / max(rate, 1e-6)))
    emb.flush()
    np.save(f["mask"], mask)
    np.save(f["pos"], pos)
    pd.DataFrame({"StudyInstanceUID": ids}).to_csv(f["ids"], index=False)
    print("{} spatial cache: {}/{} studies, {:.2f} GB, {:.0f}s".format(
        split, done, len(ids), f["emb"].stat().st_size / 1e9, time.time() - t0))
    return emb, mask, pos


if not CAN_ENCODE or DINO is None:
    raise RuntimeError("E005 requires the Kaggle DINOv2-small model and competition DICOMs")

# Test first so a runtime failure still reveals hidden-scale throughput.
emb_te, mask_te, pos_te = encode_spatial(
    test_ids, TEST_DCM, SLOTS_TEST, "test", TEST_ENCODE_BUDGET_S
)
emb_tr, mask_tr, pos_tr = encode_spatial(
    TRAIN_ENCODE_IDS, TRAIN_DCM, SLOTS_TRAIN, "train", TRAIN_ENCODE_BUDGET_S
)
train_coverage = float(mask_tr.any(axis=(1, 2)).mean())
test_coverage = float(mask_te.any(axis=(1, 2)).mean())
print("spatial coverage: train {:.2%} | test {:.2%}".format(
    train_coverage, test_coverage))

# The encoder is no longer needed; release it before training the spatial head.
DINO.cpu()
DINO = None
REGION_PROJ = REGION_PROJ.cpu()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

# Patients are 1:1 with studies in this corpus, 4,407 of 4,407, so a PatientID
# grouping is a no-op and reading DICOM PatientID to build it would be pure cost.
# The grouping that does work is duplicate report text: 49 texts are shared by 183
# studies, the largest block holding 37, and any extractor gives every study in a
# block the same twelve targets. A block split across folds scores the model on a
# target whose source it has already seen.
report_by_study = {}
if report_col is not None:
    for _sid, _txt in zip(train["StudyInstanceUID"].astype(str), train[report_col]):
        report_by_study[_sid] = _txt.strip() if isinstance(_txt, str) else ""

fold_ids = list(TRAIN_ENCODE_IDS)
group_of = {}
next_group = 0
text_group = {}
for sid in fold_ids:
    txt = report_by_study.get(sid, "")
    if txt:
        if txt not in text_group:
            text_group[txt] = next_group
            next_group += 1
        group_of[sid] = text_group[txt]
    else:
        group_of[sid] = next_group
        next_group += 1
groups = np.asarray([group_of[s] for s in fold_ids])
sizes = pd.Series(groups).value_counts()
n_blocks = int((sizes > 1).sum())
print("fold grouping: {} studies -> {} groups".format(len(fold_ids), len(sizes)))
print("  {} duplicate-report blocks covering {} studies, largest block {}".format(
    n_blocks, int(sizes[sizes > 1].sum()), int(sizes.iloc[0])))

n_splits = max(2, min(N_FOLDS, len(sizes)))
gkf = GroupKFold(n_splits=n_splits)
FOLDS = list(gkf.split(np.zeros(len(fold_ids)), groups=groups))
for k, (tr_i, va_i) in enumerate(FOLDS):
    shared = set(groups[tr_i]) & set(groups[va_i])
    assert not shared, "fold {} leaks {} groups across the split".format(k, len(shared))
print("  {} folds, no group appears on both sides of any split".format(len(FOLDS)))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import rankdata


target_idx = [LABELS.index(t) for t in SPECIAL_TARGETS]
Y = Y_soft.reindex(TRAIN_ENCODE_IDS)[SPECIAL_TARGETS].astype(np.float32).values
Y_HARD_ALL = (Y_soft.reindex(TRAIN_ENCODE_IDS)[LABELS].values >= 0.5).astype(np.int8)
raw_conf = Y_CONF.reindex(TRAIN_ENCODE_IDS)[SPECIAL_TARGETS].astype(np.float32).values
W = 0.25 + 0.75 * np.clip(raw_conf, 0.0, 1.0)
is_gold = np.asarray([s in set(gold_ids) for s in TRAIN_ENCODE_IDS], bool)
W[is_gold] = GOLD_WEIGHT


class SpatialMeniscusHead(nn.Module):
    def __init__(self):
        super().__init__()
        h = HEAD_HIDDEN
        self.norm = nn.LayerNorm(REGION_DIM)
        self.proj = nn.Linear(REGION_DIM, h)
        self.slot_emb = nn.Parameter(torch.randn(N_SLOTS, h) * 0.02)
        self.pos_proj = nn.Linear(1, h)
        self.row_emb = nn.Parameter(torch.randn(REGION_GRID, h) * 0.02)
        self.col_emb = nn.Parameter(torch.randn(REGION_GRID, h) * 0.02)
        self.query = nn.Parameter(torch.randn(len(SPECIAL_TARGETS), h) * 0.02)
        self.attn = nn.MultiheadAttention(h, HEAD_HEADS, dropout=HEAD_DROPOUT,
                                          batch_first=True)
        self.out_w = nn.Parameter(torch.randn(len(SPECIAL_TARGETS), h) * 0.02)
        self.out_b = nn.Parameter(torch.zeros(len(SPECIAL_TARGETS)))
        self.drop = nn.Dropout(HEAD_DROPOUT)
        self.out_norm = nn.LayerNorm(h)

    def forward(self, x, slice_mask, pos):
        # x: [B, slot, slice, region, dim]
        b, s, k, r, _ = x.shape
        h = self.proj(self.norm(x))
        h = h + self.slot_emb[None, :, None, None, :]
        h = h + self.pos_proj(pos[..., None, None])
        spatial = (self.row_emb[:, None, :] + self.col_emb[None, :, :]).reshape(r, -1)
        h = F.gelu(h + spatial[None, None, None, :, :])
        h = h.reshape(b, s * k * r, -1)
        valid = slice_mask[..., None].expand(-1, -1, -1, r).reshape(b, s * k * r)
        empty = ~valid.any(dim=1)
        if empty.any():
            valid = valid.clone()
            valid[empty, 0] = True
        q = self.query[None].expand(b, -1, -1)
        pooled, _ = self.attn(q, h, h, key_padding_mask=~valid, need_weights=False)
        pooled = self.drop(self.out_norm(pooled + q))
        return (pooled * self.out_w[None]).sum(-1) + self.out_b


def gather(split, rows, device):
    rows = np.sort(np.asarray(rows, dtype=int))
    emb = emb_tr if split == "train" else emb_te
    mask = mask_tr if split == "train" else mask_te
    pos = pos_tr if split == "train" else pos_te
    x = torch.from_numpy(np.asarray(emb[rows], np.float32)).to(device)
    m = torch.from_numpy(mask[rows]).to(device)
    p = torch.from_numpy(pos[rows].astype(np.float32)).to(device)
    return rows, x, m, p


@torch.no_grad()
def predict_head(model, split, rows, device):
    model.eval()
    out = np.zeros((len(rows), len(SPECIAL_TARGETS)), np.float32)
    for start in range(0, len(rows), HEAD_BATCH * 2):
        local = np.arange(start, min(len(rows), start + HEAD_BATCH * 2))
        chosen = np.asarray(rows)[local]
        ordered, x, m, p = gather(split, chosen, device)
        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=(device.type == "cuda")):
            pred = torch.sigmoid(model(x, m, p)).float().cpu().numpy()
        lookup = {int(v): j for j, v in enumerate(ordered)}
        out[local] = np.asarray([pred[lookup[int(v)]] for v in chosen])
    return out


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
oof = np.zeros((len(TRAIN_ENCODE_IDS), len(SPECIAL_TARGETS)), np.float32)
test_pred = np.zeros((len(test_ids), len(SPECIAL_TARGETS)), np.float32)
y_cpu = torch.from_numpy(Y)
w_cpu = torch.from_numpy(W)

for fold, (tr_i, va_i) in enumerate(FOLDS):
    for seed in HEAD_SEEDS:
        torch.manual_seed(seed + fold)
        model = SpatialMeniscusHead().to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WD)
        steps = math.ceil(len(tr_i) / HEAD_BATCH) * HEAD_EPOCHS
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=HEAD_LR, total_steps=steps, pct_start=0.2
        )
        rng = np.random.default_rng(seed + fold)
        for epoch in range(HEAD_EPOCHS):
            model.train()
            order = rng.permutation(tr_i)
            running = 0.0
            for start in range(0, len(order), HEAD_BATCH):
                chosen = order[start:start + HEAD_BATCH]
                ordered, x, m, p = gather("train", chosen, device)
                with torch.autocast(device_type="cuda", dtype=torch.float16,
                                    enabled=(device.type == "cuda")):
                    logits = model(x, m, p)
                yy = y_cpu[ordered].to(device)
                ww = w_cpu[ordered].to(device)
                loss_el = F.binary_cross_entropy_with_logits(
                    logits.float(), yy, reduction="none"
                )
                loss = (loss_el * ww).sum() / ww.sum().clamp_min(1.0)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
                sched.step()
                running += float(loss.detach())
            print("fold {} epoch {}/{} loss {:.4f}".format(
                fold, epoch + 1, HEAD_EPOCHS,
                running / max(1, math.ceil(len(order) / HEAD_BATCH))))
        oof[va_i] += predict_head(model, "train", va_i, device) / len(HEAD_SEEDS)
        test_pred += predict_head(
            model, "test", np.arange(len(test_ids)), device
        ) / (len(FOLDS) * len(HEAD_SEEDS))
        torch.save({
            "format": 1,
            "fold": fold,
            "seed": seed,
            "targets": SPECIAL_TARGETS,
            "cache_version": SPATIAL_CACHE_VERSION,
            "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        }, WORK / "e005_meniscus_fold{}_seed{}.pt".format(fold, seed))
        del model, opt, sched
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def find_anchor_oof():
    required = {"ids", "pred", "y_derived", "gold_mask", "targets"}
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in
                   ("competitions", "train_series", "test_series", "__pycache__")]
        if "oof.npz" in files:
            path = Path(root) / "oof.npz"
            try:
                z = np.load(path, allow_pickle=False)
                if required.issubset(z.files):
                    hits.append(path)
            except Exception:
                pass
    hits.sort(key=lambda p: ("rsna-knee-weights" not in str(p), str(p)))
    if not hits:
        raise FileNotFoundError("anchor oof.npz not found")
    return hits[0]


def rank_columns(a):
    return np.stack([
        (rankdata(a[:, j], method="average") - 1.0) / max(1, len(a) - 1)
        for j in range(a.shape[1])
    ], axis=1)


def auc_columns(y, p):
    return np.asarray([roc_auc_score(y[:, j], p[:, j]) for j in range(y.shape[1])])


anchor_path = find_anchor_oof()
anchor_sha = hashlib.sha256(anchor_path.read_bytes()).hexdigest()
if anchor_sha != EXPECTED_ANCHOR_OOF_SHA256:
    raise RuntimeError("anchor OOF hash drift")
anchor = np.load(anchor_path, allow_pickle=False)
anchor_targets = [str(x) for x in anchor["targets"]]
anchor_cols = [anchor_targets.index(t) for t in SPECIAL_TARGETS]
ours_pos = {s: i for i, s in enumerate(TRAIN_ENCODE_IDS)}
anchor_pos = {str(s): i for i, s in enumerate(anchor["ids"])}
common = [str(s) for s in anchor["ids"] if str(s) in ours_pos]
oi = np.asarray([ours_pos[s] for s in common], int)
ai = np.asarray([anchor_pos[s] for s in common], int)
if len(common) != len(TRAIN_ENCODE_IDS):
    raise RuntimeError("specialist and anchor OOF study coverage differ")

y_ref = np.asarray(anchor["y_derived"][ai][:, anchor_cols], np.int8)
gold_mask_ref = np.asarray(anchor["gold_mask"][ai], bool)
gold_y = Y_HARD_ALL[oi][:, target_idx]
anchor_pred = np.asarray(anchor["pred"][ai][:, anchor_cols], np.float64)
arm_pred = np.asarray(oof[oi], np.float64)
anchor_rank = rank_columns(anchor_pred)
arm_rank = rank_columns(arm_pred)
candidate = (1.0 - SPECIALIST_BLEND_W) * anchor_rank + SPECIALIST_BLEND_W * arm_rank

anchor_auc = auc_columns(y_ref, anchor_rank)
arm_auc = auc_columns(y_ref, arm_rank)
candidate_auc = auc_columns(y_ref, candidate)
delta = candidate_auc - anchor_auc
anchor_gold = auc_columns(gold_y[gold_mask_ref], anchor_rank[gold_mask_ref])
arm_gold = auc_columns(gold_y[gold_mask_ref], arm_rank[gold_mask_ref])
candidate_gold = auc_columns(gold_y[gold_mask_ref], candidate[gold_mask_ref])
corr = np.asarray([
    np.corrcoef(anchor_rank[:, j], arm_rank[:, j])[0, 1]
    for j in range(len(SPECIAL_TARGETS))
])

per_target = [
    {
        "target": target,
        "anchor_auc": float(anchor_auc[j]),
        "arm_auc": float(arm_auc[j]),
        "candidate_auc": float(candidate_auc[j]),
        "candidate_delta": float(delta[j]),
        "anchor_gold_auc": float(anchor_gold[j]),
        "arm_gold_auc": float(arm_gold[j]),
        "candidate_gold_auc": float(candidate_gold[j]),
        "spearman": float(corr[j]),
    }
    for j, target in enumerate(SPECIAL_TARGETS)
]
lat = SPECIAL_TARGETS.index("Lateral Meniscus")
med = SPECIAL_TARGETS.index("Medial Meniscus")
checks = {
    "cache_coverage": train_coverage >= 0.99,
    "mean_gain": float(delta.mean()) >= GATE_MEAN_GAIN,
    "lateral_gain": float(delta[lat]) >= GATE_LATERAL_GAIN,
    "medial_floor": float(delta[med]) >= GATE_MEDIAL_FLOOR,
    "gold_direction": float((candidate_gold - anchor_gold).mean()) >= GATE_GOLD_FLOOR,
    "diversity": float(np.median(corr)) <= GATE_MAX_CORR,
}
gate = {
    "pass": bool(all(checks.values())),
    "decision": "expand_specialist" if all(checks.values()) else "stop_specialist",
    "checks": checks,
    "cache_version": SPATIAL_CACHE_VERSION,
    "train_coverage": train_coverage,
    "test_coverage": test_coverage,
    "n_studies": len(common),
    "n_gold": int(gold_mask_ref.sum()),
    "specialist_weight": SPECIALIST_BLEND_W,
    "mean_anchor_auc": float(anchor_auc.mean()),
    "mean_arm_auc": float(arm_auc.mean()),
    "mean_candidate_auc": float(candidate_auc.mean()),
    "mean_gain": float(delta.mean()),
    "mean_anchor_gold_auc": float(anchor_gold.mean()),
    "mean_arm_gold_auc": float(arm_gold.mean()),
    "mean_candidate_gold_auc": float(candidate_gold.mean()),
    "mean_gold_gain": float((candidate_gold - anchor_gold).mean()),
    "median_spearman": float(np.median(corr)),
    "per_target": per_target,
}

print(pd.DataFrame(per_target).round(5).to_string(index=False))
print("E005 GATE:", "GO" if gate["pass"] else "NO-GO", gate["decision"], checks)
(WORK / "e005_gate.json").write_text(json.dumps(gate, indent=2, sort_keys=True))
np.savez_compressed(
    WORK / "e005_oof_predictions.npz",
    ids=np.asarray(common, str), targets=np.asarray(SPECIAL_TARGETS, str),
    y=y_ref, gold_y=gold_y, gold_mask=gold_mask_ref,
    anchor=np.asarray(anchor_pred, np.float32), arm=np.asarray(arm_pred, np.float32),
    candidate=np.asarray(candidate, np.float32),
)
np.savez_compressed(
    WORK / "e005_test_predictions.npz",
    ids=np.asarray(test_ids, str), targets=np.asarray(SPECIAL_TARGETS, str),
    arm=np.asarray(test_pred, np.float32),
)

# Valid diagnostic file only; this run is not submitted to the leaderboard.
sub = sample_sub.copy() if len(sample_sub) == len(test) else test[["StudyInstanceUID"]].copy()
for label in LABELS:
    sub[label] = 0.5
if len(test_pred) > 1:
    ranked_test = rank_columns(test_pred)
    for j, label in enumerate(SPECIAL_TARGETS):
        sub[label] = ranked_test[:, j]
sub.to_csv(WORK / "submission.csv", index=False)
print("wrote E005 artifacts and non-submitted diagnostic submission.csv")
